## Load data

In [36]:
import json
import numpy as np

with open("tallinn_osrm_matrix.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Depo + 4 piegādes
n = 5

names = [p["name"] for p in data["points"][:n]]
distances = np.array(data["distances"], dtype=float)[:n, :n] / 1000

DEPOT = 0
DELIVERIES = list(range(1, n))
COURIERS = 2

print("Depot:", names[DEPOT])
print("Deliveries:", len(DELIVERIES))
print("Couriers:", COURIERS)

for i, name in enumerate(names):
    print(f"{i}: {name}")

Depot: Omniva Pallasti (depot)
Deliveries: 4
Couriers: 2
0: Omniva Pallasti (depot)
1: Kadriorg President's Office
2: Shaurma Kebab Punane
3: Kultuurikatel
4: Patarei Prison


## QUBO variables

In [38]:
ACTIVE_DELIVERIES = DELIVERIES[:3]

variables = {}
index = 0

for courier in range(COURIERS):
    for position in range(len(ACTIVE_DELIVERIES)):
        for delivery in ACTIVE_DELIVERIES:
            variables[(courier, position, delivery)] = index
            index += 1

NUM_QUBITS = index

print("Deliveries:", ACTIVE_DELIVERIES)
print("QUBO variables:", len(variables))
print("Required qubits:", NUM_QUBITS)

Deliveries: [1, 2, 3]
QUBO variables: 18
Required qubits: 18


## Build QUBO Constraints

In [39]:
import numpy as np

Q = np.zeros((NUM_QUBITS, NUM_QUBITS))
PENALTY = 100.0

m = len(ACTIVE_DELIVERIES)


def add_squared_constraint(terms, target, weight):
    """
    Add weight * (sum(a*x) - target)^2 to the QUBO.
    Constant terms are omitted.
    """
    for i, a in terms:
        Q[i, i] += weight * (a * a - 2 * target * a)

    for idx1 in range(len(terms)):
        i, a = terms[idx1]

        for idx2 in range(idx1 + 1, len(terms)):
            j, b = terms[idx2]
            Q[i, j] += 2 * weight * a * b


# 1. Each delivery must be visited exactly once
for delivery in ACTIVE_DELIVERIES:
    terms = [
        (variables[(courier, position, delivery)], 1)
        for courier in range(COURIERS)
        for position in range(m)
    ]

    add_squared_constraint(terms, 1, PENALTY)


# 2. At most one delivery per route position
for courier in range(COURIERS):
    for position in range(m):
        for a in range(m):
            for b in range(a + 1, m):
                i = variables[
                    (courier, position, ACTIVE_DELIVERIES[a])
                ]
                j = variables[
                    (courier, position, ACTIVE_DELIVERIES[b])
                ]

                Q[i, j] += PENALTY


# 3. No empty positions before occupied positions
# Penalize occupancy(p+1) * (1 - occupancy(p))
for courier in range(COURIERS):
    for position in range(m - 1):
        for next_delivery in ACTIVE_DELIVERIES:
            nxt = variables[
                (courier, position + 1, next_delivery)
            ]

            Q[nxt, nxt] += PENALTY

            for current_delivery in ACTIVE_DELIVERIES:
                cur = variables[
                    (courier, position, current_delivery)
                ]

                Q[cur, nxt] -= PENALTY


# 4. Both couriers must receive at least one delivery
# Require the first position of each courier to be occupied
for courier in range(COURIERS):
    terms = [
        (variables[(courier, 0, delivery)], 1)
        for delivery in ACTIVE_DELIVERIES
    ]

    add_squared_constraint(terms, 1, PENALTY)


print("QUBO matrix shape:", Q.shape)
print("Nonzero QUBO coefficients:", np.count_nonzero(Q))

QUBO matrix shape: (18, 18)
Nonzero QUBO coefficients: 93


## Add route distance costs

In [40]:
# Add travel distances to the QUBO objective

for courier in range(COURIERS):

    # Distance from depot to the first delivery
    for delivery in ACTIVE_DELIVERIES:
        i = variables[(courier, 0, delivery)]
        Q[i, i] += distances[DEPOT, delivery]

    # Distance between consecutive deliveries
    for position in range(m - 1):
        for delivery_a in ACTIVE_DELIVERIES:
            for delivery_b in ACTIVE_DELIVERIES:
                i = variables[(courier, position, delivery_a)]
                j = variables[(courier, position + 1, delivery_b)]

                Q[i, j] += distances[delivery_a, delivery_b]

    # Distance from the last delivery back to depot
    # A delivery is the last stop if the next position is empty
    for position in range(m):
        for delivery in ACTIVE_DELIVERIES:
            i = variables[(courier, position, delivery)]

            Q[i, i] += distances[delivery, DEPOT]

            if position < m - 1:
                for next_delivery in ACTIVE_DELIVERIES:
                    j = variables[(courier, position + 1, next_delivery)]
                    Q[i, j] -= distances[delivery, DEPOT]


print("QUBO matrix shape:", Q.shape)
print("Nonzero QUBO coefficients:", np.count_nonzero(Q))

QUBO matrix shape: (18, 18)
Nonzero QUBO coefficients: 105


## Validate all feasible routes

In [42]:
from itertools import permutations

constant_offset = PENALTY * (len(ACTIVE_DELIVERIES) + COURIERS)

def route_distance(route):
    full_route = [DEPOT] + list(route) + [DEPOT]
    return sum(
        distances[a, b]
        for a, b in zip(full_route[:-1], full_route[1:])
    )


tested = 0
mismatches = []

# Assign each delivery to one of two couriers
for mask in range(1, 2**m - 1):
    group1 = [
        d for i, d in enumerate(ACTIVE_DELIVERIES)
        if not (mask & (1 << i))
    ]
    group2 = [
        d for i, d in enumerate(ACTIVE_DELIVERIES)
        if mask & (1 << i)
    ]

    # Test every possible visiting order
    for route1 in permutations(group1):
        for route2 in permutations(group2):
            x = np.zeros(NUM_QUBITS)

            for courier, route in enumerate([route1, route2]):
                for position, delivery in enumerate(route):
                    x[variables[(courier, position, delivery)]] = 1

            qubo_cost = x @ Q @ x + constant_offset
            actual_cost = route_distance(route1) + route_distance(route2)

            tested += 1

            if not np.isclose(qubo_cost, actual_cost):
                mismatches.append((route1, route2, qubo_cost, actual_cost))

print("Routes tested:", tested)
print("Mismatches:", len(mismatches))
print("All costs match:", len(mismatches) == 0)

Routes tested: 12
Mismatches: 0
All costs match: True


## Convert QUBO to Ising Hamiltonian

In [43]:
from qiskit.quantum_info import SparsePauliOp

pauli_terms = []
constant = 0.0

# QUBO: x_i = (I - Z_i) / 2
for i in range(NUM_QUBITS):
    for j in range(i, NUM_QUBITS):
        coefficient = Q[i, j]

        if abs(coefficient) < 1e-12:
            continue

        if i == j:
            constant += coefficient / 2

            label = ["I"] * NUM_QUBITS
            label[NUM_QUBITS - 1 - i] = "Z"

            pauli_terms.append(
                ("".join(label), -coefficient / 2)
            )

        else:
            constant += coefficient / 4

            for qubit in (i, j):
                label = ["I"] * NUM_QUBITS
                label[NUM_QUBITS - 1 - qubit] = "Z"

                pauli_terms.append(
                    ("".join(label), -coefficient / 4)
                )

            label = ["I"] * NUM_QUBITS
            label[NUM_QUBITS - 1 - i] = "Z"
            label[NUM_QUBITS - 1 - j] = "Z"

            pauli_terms.append(
                ("".join(label), coefficient / 4)
            )


cost_operator = SparsePauliOp.from_list(
    pauli_terms
).simplify()

# Scale the Hamiltonian for QAOA optimization
SCALE = 100.0
cost_operator = cost_operator / SCALE

print("Qubits:", cost_operator.num_qubits)
print("Pauli terms:", len(cost_operator))
print(f"Constant offset: {constant:.3f}")

Qubits: 18
Pauli terms: 105
Constant offset: 1538.719


## QAOA cost function

In [48]:
from scipy.linalg import hadamard

# Precompute the energy of every computational basis state
NUM_STATES = 2 ** NUM_QUBITS

states = np.arange(NUM_STATES, dtype=np.uint32)

# Extract binary values for all QUBO variables
bits = ((states[:, None] >> np.arange(NUM_QUBITS)) & 1).astype(float)

# Calculate Ising energies, excluding the constant offset
energies = np.einsum("bi,ij,bj->b", bits, Q, bits)
energies = (energies - constant) / SCALE

# Free memory
del bits

def qaoa_energy(params):
    gamma_value, beta_value = params

    # Start in uniform superposition
    amplitudes = np.ones(NUM_STATES, dtype=complex) / np.sqrt(NUM_STATES)

    # Apply the QAOA cost unitary
    amplitudes *= np.exp(-1j * gamma_value * energies)

    # Apply the RX mixer to every qubit
    for qubit in range(NUM_QUBITS):
        step = 1 << qubit
        amplitudes = amplitudes.reshape(-1, 2, step)

        a = amplitudes[:, 0, :].copy()
        b = amplitudes[:, 1, :].copy()

        amplitudes[:, 0, :] = np.cos(beta_value) * a - 1j * np.sin(beta_value) * b
        amplitudes[:, 1, :] = np.cos(beta_value) * b - 1j * np.sin(beta_value) * a

        amplitudes = amplitudes.reshape(-1)

    # Calculate expected energy
    probabilities = np.abs(amplitudes) ** 2
    return float(np.dot(probabilities, energies))


initial_params = [0.5, 0.5]

energy = qaoa_energy(initial_params)

print(f"Initial QAOA energy: {energy:.6f}")
print("Energy is finite:", np.isfinite(energy))

Initial QAOA energy: 12.072950
Energy is finite: True


## Optimize QAOA Parameters

In [49]:
from scipy.optimize import minimize
import time

# Track optimization progress
iteration = 0
start_time = time.time()

def objective(params):
    return qaoa_energy(params)

def callback(params):
    global iteration
    iteration += 1

    energy = qaoa_energy(params)

    print(
        f"Iteration {iteration}: "
        f"energy = {energy:.6f}"
    )

# Optimize QAOA parameters
result = minimize(
    objective,
    x0=[0.5, 0.5],
    method="COBYLA",
    callback=callback,
    options={
        "maxiter": 30,
        "rhobeg": 0.3
    }
)

optimal_gamma, optimal_beta = result.x

print("\nOptimization finished!")
print(f"Optimal gamma: {optimal_gamma:.6f}")
print(f"Optimal beta: {optimal_beta:.6f}")
print(f"Final QAOA energy: {result.fun:.6f}")
print(f"Execution time: {time.time() - start_time:.2f} seconds")

Iteration 1: energy = -1.407810
Iteration 2: energy = -1.465994
Iteration 3: energy = -1.465994
Iteration 4: energy = -1.775527
Iteration 5: energy = -1.920297
Iteration 6: energy = -1.920297
Iteration 7: energy = -1.942811
Iteration 8: energy = -1.942811
Iteration 9: energy = -1.942811
Iteration 10: energy = -1.947086
Iteration 11: energy = -1.947086
Iteration 12: energy = -1.947093
Iteration 13: energy = -1.947114
Iteration 14: energy = -1.947114
Iteration 15: energy = -1.947141
Iteration 16: energy = -1.947169
Iteration 17: energy = -1.947197
Iteration 18: energy = -1.947200
Iteration 19: energy = -1.947224
Iteration 20: energy = -1.947227
Iteration 21: energy = -1.947230
Iteration 22: energy = -1.947230

Optimization finished!
Optimal gamma: 0.794934
Optimal beta: 0.330698
Final QAOA energy: -1.947230
Execution time: 6.29 seconds


## Extract QAOA Routes

In [50]:
# Reconstruct the optimized QAOA state

gamma_value, beta_value = result.x

amplitudes = np.ones(NUM_STATES, dtype=complex) / np.sqrt(NUM_STATES)

# Apply the cost unitary
amplitudes *= np.exp(-1j * gamma_value * energies)

# Apply the mixer
for qubit in range(NUM_QUBITS):
    step = 1 << qubit
    amplitudes = amplitudes.reshape(-1, 2, step)

    a = amplitudes[:, 0, :].copy()
    b = amplitudes[:, 1, :].copy()

    amplitudes[:, 0, :] = np.cos(beta_value) * a - 1j * np.sin(beta_value) * b
    amplitudes[:, 1, :] = np.cos(beta_value) * b - 1j * np.sin(beta_value) * a

    amplitudes = amplitudes.reshape(-1)

probabilities = np.abs(amplitudes) ** 2

# Inspect all valid route pairs
valid_solutions = []

for state_index in range(NUM_STATES):
    x = ((state_index >> np.arange(NUM_QUBITS)) & 1)

    routes = []

    for courier in range(COURIERS):
        route = []

        for position in range(m):
            for delivery in ACTIVE_DELIVERIES:
                if x[variables[(courier, position, delivery)]]:
                    route.append(delivery)

        routes.append(route)

    # Check all deliveries are assigned exactly once
    visited = routes[0] + routes[1]

    if (
        len(routes[0]) > 0
        and len(routes[1]) > 0
        and sorted(visited) == sorted(ACTIVE_DELIVERIES)
        and all(
            sum(x[variables[(c, p, d)]] for d in ACTIVE_DELIVERIES) <= 1
            for c in range(COURIERS)
            for p in range(m)
        )
        and all(
            not (
                sum(x[variables[(c, p, d)]] for d in ACTIVE_DELIVERIES) == 0
                and sum(x[variables[(c, p + 1, d)]] for d in ACTIVE_DELIVERIES) > 0
            )
            for c in range(COURIERS)
            for p in range(m - 1)
        )
    ):
        valid_solutions.append((
            probabilities[state_index],
            routes
        ))

valid_solutions.sort(reverse=True, key=lambda item: item[0])

print("Valid solutions:", len(valid_solutions))

if valid_solutions:
    probability, best_routes = valid_solutions[0]

    total_distance = 0.0

    for courier, route in enumerate(best_routes):
        distance = route_distance(route)
        total_distance += distance

        print(
            f"Courier {courier + 1}: "
            f"{[DEPOT] + route + [DEPOT]} "
            f"({distance:.3f} km)"
        )

    print(f"Total distance: {total_distance:.3f} km")
    print(f"Solution probability: {probability:.6%}")

Valid solutions: 12
Courier 1: [0, 1, 3, 0] (10.116 km)
Courier 2: [0, 2, 0] (5.515 km)
Total distance: 15.631 km
Solution probability: 0.000061%


## QAOA simulation

In [51]:
# Simulate quantum measurements from the optimized QAOA state

SHOTS = 10000
rng = np.random.default_rng(42)

samples = rng.choice(
    NUM_STATES,
    size=SHOTS,
    p=probabilities / probabilities.sum()
)

valid_states = []

for state_index in samples:
    x = ((int(state_index) >> np.arange(NUM_QUBITS)) & 1)

    routes = []

    for courier in range(COURIERS):
        route = []

        for position in range(m):
            selected = [
                delivery
                for delivery in ACTIVE_DELIVERIES
                if x[variables[(courier, position, delivery)]]
            ]

            if len(selected) > 1:
                break

            if selected:
                route.append(selected[0])

        routes.append(route)

    if (
        len(routes) == COURIERS
        and all(len(route) > 0 for route in routes)
        and sorted(routes[0] + routes[1]) == sorted(ACTIVE_DELIVERIES)
        and all(
            sum(x[variables[(c, p, d)]] for d in ACTIVE_DELIVERIES) <= 1
            for c in range(COURIERS)
            for p in range(m)
        )
        and all(
            sum(x[variables[(c, p + 1, d)]] for d in ACTIVE_DELIVERIES)
            <= sum(x[variables[(c, p, d)]] for d in ACTIVE_DELIVERIES)
            for c in range(COURIERS)
            for p in range(m - 1)
        )
    ):
        valid_states.append((state_index, routes))

print("Total shots:", SHOTS)
print("Valid measurements:", len(valid_states))
print(f"Feasibility rate: {len(valid_states) / SHOTS:.4%}")

if valid_states:
    best_state, best_routes = min(
        valid_states,
        key=lambda item: sum(route_distance(r) for r in item[1])
    )

    print("Best sampled routes:", best_routes)
    print(
        "Best sampled distance:",
        round(sum(route_distance(r) for r in best_routes), 3),
        "km"
    )

Total shots: 10000
Valid measurements: 0
Feasibility rate: 0.0000%
